In [ ]:
# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import matplotlib.cm as cm
import matplotlib.patheffects as pe

# ======================== VARIABLES ========================
# 起点和终点的节点 ID（按需修改）
source_id      = 13
destination_id = 17

# ======================== CSV DIRECTORY ========================
# 假设所有 CSV 文件都在当前工作目录
csv_dir = os.getcwd()

# ======================== LOAD SHADOW PROFILES ========================
shadow_profiles = {}
# 查找所有 tree 文件并加载对应 shadow 文件
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # 去掉 '_tree.csv'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    # 读取数值矩阵
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # 如 ['edge','0','1'] → ['0','1']
    shadow_profiles[f"edge_{v}_{u}"] = combined

# ======================== LOAD SOLAR & CLOUD ========================
# 读取实际时间序列
solar_file = os.path.join(csv_dir, 'solar_intensity.csv')
cloud_file = os.path.join(csv_dir, 'cloud_coverage.csv')
if not os.path.exists(solar_file) or not os.path.exists(cloud_file):
    raise FileNotFoundError("缺少 'solar_intensity.csv' 或 'cloud_coverage.csv' 文件")
solar_df = pd.read_csv(solar_file)
cloud_df = pd.read_csv(cloud_file)
R = solar_df.iloc[:,2].astype(float).values
C = cloud_df.iloc[:,2].astype(float).values

# 时间步长
T = R.shape[0]

# ======================== NODE COORDS & ID MAPPING ========================
node_coords = {
    0: (0, 0),         # Downtown center
    1: (0.67, 0.42),   # Northeast of downtown
    2: (0.33, 0.89),   # North of downtown
    3: (-0.48, 0.78),  # Northwest of downtown
    4: (-0.92, 0.31),  # West of downtown
    5: (-0.75, -0.56), # Southwest of downtown
    6: (-0.37, -0.83), # South of downtown
    7: (0.52, -0.69),  # Southeast of downtown
    8: (0.86, -0.27),  # East of downtown
    9: (1.73, 0.86),   # Northeast suburb
    10: (1.26, 1.87),  # North suburb
    11: (-1.32, 1.76), # Northwest suburb
    12: (-1.89, 0.72), # West suburb
    13: (-1.65, -0.94),# Southwest suburb
    14: (-1.18, -1.73),# South suburb
    15: (1.31, -1.62), # Southeast suburb
    16: (1.81, -0.83), # East suburb
    17: (2.23, 0.21),  # Far east
    18: (0.18, 1.76),  # Far north
    19: (0.11, -1.65)  # Far south
}

coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# ======================== RAW EDGES (COORDINATE-BASED) ========================
raw_edges = {
    # ——— from center to first ring ———
    ((0, 0),       (0.67, 0.42)):     40,
    ((0, 0),       (0.33, 0.89)):     55,
    ((0, 0),       (-0.48, 0.78)):    58,
    ((0, 0),       (-0.92, 0.31)):    54,
    ((0, 0),       (-0.75, -0.56)):   52,
    ((0, 0),       (-0.37, -0.83)):   57,
    ((0, 0),       (0.52, -0.69)):    53,
    ((0, 0),       (0.86, -0.27)):    57,

    # ——— first-ring cycle ———
    ((0.67, 0.42), (0.33, 0.89)):     27,
    ((0.33, 0.89), (-0.48, 0.78)):    45,
    ((-0.48, 0.78),(-0.92, 0.31)):    35,
    ((-0.92, 0.31),(-0.75, -0.56)):   54,
    ((-0.75, -0.56),(-0.37, -0.83)):  20,
    ((-0.37, -0.83),(0.52, -0.69)):   54,
    ((0.52, -0.69),(0.86, -0.27)):    26,
    ((0.86, -0.27),(0.67, 0.42)):     42,

    # ——— spokes to second ring ———
    ((0.67, 0.42),  (1.73, 0.86)):    73,   # 1 → 9
    ((0.33, 0.89),  (0.18, 1.76)):    56,   # 2 → 18
    ((-0.48, 0.78), (-1.32, 1.76)):   87,   # 3 → 11
    ((-0.92, 0.31), (-1.89, 0.72)):   59,   # 4 → 12
    ((-0.75, -0.56),(-1.65, -0.94)):  54,   # 5 → 13
    ((-0.37, -0.83),(-1.18, -1.73)):  80,   # 6 → 14
    ((0.52, -0.69), (1.31, -1.62)):   71,   # 7 → 15
    ((0.86, -0.27), (1.81, -0.83)):   70,   # 8 → 16

    # ——— second-ring cycle (adjusted 9–16–17) ———
    ((1.26, 1.87),  (1.73, 0.86)):    74,   # 10 ↔ 9
    ((1.73, 0.86),  (2.23, 0.21)):    46,   # 9 ↔ 17
    ((1.81, -0.83), (2.23, 0.21)):    72,   # 16 ↔ 17
    # (removed original 17 ↔ 9 link)

    # ——— remaining second-ring connections ———
    ((1.26, 1.87),  (0.18, 1.76)):    67,   # 10 ↔ 18
    ((0.18, 1.76),  (-1.32, 1.76)):   89,   # 18 ↔ 11
    ((-1.32, 1.76), (-1.89, 0.72)):   75,   # 11 ↔ 12
    ((-1.89, 0.72), (-1.65, -0.94)):  110,   # 12 ↔ 13
    ((-1.65, -0.94),(-1.18, -1.73)):  55,   # 13 ↔ 14
    ((-1.18, -1.73),(0.11, -1.65)):   74,   # 14 ↔ 19
    ((0.11, -1.65), (1.31, -1.62)):   79,   # 19 ↔ 15
    ((1.31, -1.62), (1.81, -0.83)):   56    # 15 ↔ 16
}



# ======================== DUMMY SHADOW PROFILE FILL ========================
# 对缺失 shadow profile 的边，用 shape=(T,1) 的低值 dummy 填充
for (cu, cv), _ in raw_edges.items():
    u_id = coord_to_id[cu]
    v_id = coord_to_id[cv]
    p = f"edge_{u_id}_{v_id}"; r = f"edge_{v_id}_{u_id}"
    if p not in shadow_profiles:
        dummy = np.full((T,1), -999999.0)
        shadow_profiles[p] = dummy
        shadow_profiles[r] = dummy

# ======================== GRAPH SETUP (ALL EDGES) ========================
graph = {}
for (cu, cv), dist in raw_edges.items():
    u_id = coord_to_id[cu]; v_id = coord_to_id[cv]
    graph.setdefault(u_id, {})[v_id] = dist
    graph.setdefault(v_id, {})[u_id] = dist

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

# ======================== SOLAR EXPOSURE CALC ========================
def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"
    profile = shadow_profiles[key]
    _, K = profile.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        # z < 0 会引入高惩罚：1 - z ≈ 1e6
        z = profile[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path: continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, t0 + steps))
    return results

# ======================== RUN & DISPLAY ========================
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)

In [ ]:

# ======================== PARETO FRONTIER PLOT (美化) ========================
# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]
# ——— Extract the two objective lists ———
ds = [path_obj[1][0] for path_obj in pareto]   # total distances
ss = [path_obj[1][1] for path_obj in pareto]   # total solar exposures

# 1) 计算均值
mean_d = sum(ds) / len(ds)
mean_s = sum(ss) / len(ss)

# 2) 找到三个特殊点索引
idx_shortest   = min(range(len(ds)), key=lambda i: ds[i])
idx_best_shade = min(range(len(ss)), key=lambda i: ss[i])
idx_compromise = min(
    range(len(ds)),
    key=lambda i: (ds[i] - mean_d)**2 + (ss[i] - mean_s)**2
)

# 3) 开始绘图
plt.figure(figsize=(10, 6))

# 所有 Pareto 点
plt.scatter(ds, ss,
            marker='x',
            s=400,
            color='tab:blue',
            label='Pareto-optimal paths')

# 高亮设置：索引, 标签, 颜色, 垂直对齐, 水平对齐
highlights = [
    (idx_shortest,  'Shortest Path',        'tab:green',  'bottom', 'left'),
    (idx_best_shade, 'Best Shade',           'tab:orange', 'top',    'right'),
    (idx_compromise, 'Balanced Compromise',  'tab:red',    'bottom', 'right'),
]

for idx, label, color, va, ha in highlights:
    plt.scatter(ds[idx], ss[idx],
                marker='X',
                s=800,
                color=color,
                linewidths=0.8,   # 控制 “×” 的粗细，数值越小越细
                label=label)

# 标题与坐标轴
plt.title("Pareto Frontier: Distance vs. Solar Exposure", fontsize=16, pad=12)
plt.xlabel("Total Distance", fontsize=14)
plt.ylabel("Total Solar Exposure", fontsize=14)

# 网格
plt.grid(linestyle='--', alpha=0.5)

# 美观的图例：放图内右上角，略偏移避免重叠
leg = plt.legend(
    loc='upper right',        # 图例位置
    fontsize=16,              # 文字大小
    frameon=True,             # 显示边框
    borderpad=1.0,            # 边框内间距
    labelspacing=1.0,         # 条目之间间隔
    handlelength=2.5,         # 图例句柄长度
    handletextpad=1.0,        # 图例句柄与文字间距
    markerscale=0.8           # 放大标记尺寸
)
leg.get_frame().set_edgecolor('gray')   # 边框颜色
leg.get_frame().set_alpha(0.9)          # 背景透明度


# 留出图例空间
plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

In [ ]:
# ======================== DRAW & ROUTE ========================
# 假定 graph, node_coords, pareto 都已定义

fig, ax = plt.subplots(figsize=(14, 12))

# 1) 背景路段（浅灰色）
seen = set()
for u, nbrs in graph.items():
    for v, dist in nbrs.items():
        if (v, u) in seen: continue
        seen.add((u, v))
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                color='lightgray', linewidth=1, zorder=1)

# 2) 节点：先画一个白底黑边的大点，再加粗数字
for nid, (x, y) in node_coords.items():
    # 用黄色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为黄色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 3) Pareto 路径：不同线型 + 半透明 + 颜色
cmap = cm.get_cmap('tab10', len(pareto))
line_styles = ['solid', 'dashed', 'dotted', 'dashdot']
for i, (path, (dist, sol)) in enumerate(pareto):
    pts = [node_coords[n] for n in path]
    xs, ys = zip(*pts)
    ax.plot(xs, ys,
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=8,
            linewidth=2.5,
            alpha=0.9,
            color=cmap(i),
            label=f'Path {i+1}: d={dist:.0f}, s={sol:.1f}',
            zorder=4)

# 4) 图例移动到画布外，并优化布局
ax.legend(loc='upper left',
          bbox_to_anchor=(1.02, 1),
          fontsize='small',
          frameon=True)

# 美化
ax.set_title("Pareto-optimal Paths on Pedestrian Network", fontsize=16)
ax.set_xlabel("X Coordinate", fontsize=14)
ax.set_ylabel("Y Coordinate", fontsize=14)
ax.set_aspect('equal', 'box')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
ax = plt.gca()

for (u, v), shade in edge_shade.items():
    if u < v:
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                linewidth=8,
                solid_capstyle='butt',
                color=plt.cm.YlGn(shade),  # 0→yellow, 1→green
                alpha=0.8)

# 节点
xs, ys = zip(*node_coords.values())
ax.scatter(xs, ys, s=50, color='white', edgecolor='black', zorder=2)

for nid, (x, y) in node_coords.items():
    # 用黄色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为黄色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 色标
sm = plt.cm.ScalarMappable(cmap='YlGn', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Average Shade (0=no, 1=full)', fontsize=12)

ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Road Segment Shade Heatmap", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

# 假设 raw_edges, node_coords, shadow_profiles, coord_to_id, graph 等变量
# 已经在脚本中定义并计算了 edge_shade（每条边的平均 shade）

# 1) 计算每个节点的平均 shade 值
node_shade = {nid: [] for nid in node_coords}
for (u, v), shade in edge_shade.items():
    node_shade[u].append(shade)
    node_shade[v].append(shade)
# 只保留平均值
node_vals = {nid: np.mean(vals) for nid, vals in node_shade.items()}

# 2) 准备三角剖分输入
xs = np.array([node_coords[n][0] for n in node_coords])
ys = np.array([node_coords[n][1] for n in node_coords])
zs = np.array([node_vals[n] for n in node_coords])

triang = mtri.Triangulation(xs, ys)

# 3) 绘制等高线填色图
plt.figure(figsize=(10, 8))
contf = plt.tricontourf(triang, zs, levels=12, cmap='YlGn', alpha=0.8)
cont = plt.tricontour(triang, zs, levels=12, colors='k', linewidths=0.5)

# 4) 叠加网络边
for u, nbrs in graph.items():
    for v in nbrs:
        if u < v:
            x1, y1 = node_coords[u]
            x2, y2 = node_coords[v]
            plt.plot([x1, x2], [y1, y2], color='gray', linewidth=1, alpha=0.6)

# 5) 叠加节点
for nid, (x, y) in node_coords.items():
    plt.scatter(x, y, s=40, facecolor='white', edgecolor='black', linewidth=0.8, zorder=3)
    plt.text(x, y+0.02, str(nid), ha='center', va='bottom', fontsize=9)

plt.colorbar(contf, label='Average Shade (0=no, 1=full)')
plt.title("Shadow Intensity Contour Map")
plt.axis('equal')
plt.xticks([]); plt.yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# …（前面数据准备完毕）…

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 光滑曲面，cmap='YlGn'：0→yellow，1→green
surf = ax.plot_surface(
    Xi, Yi, Zi,
    cmap='YlGn',
    edgecolor='none',
    antialiased=True,
    rcount=10000, ccount=10000  # 更细网格
)

# 原始节点叠加
ax.scatter(xs, ys, zs, color='black', s=30, zorder=5)

# **调整视角**：抬高视点，增大俯仰，让曲面更“倾斜”
ax.view_init(elev=45, azim=-60)

# 坐标轴标签和标题
ax.set_title("Smooth 3D Shadow Surface", fontsize=16)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_zlabel("Average Shade (0–1)")

# 色标
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Average Shade')
plt.tight_layout()
plt.show()


In [ ]:
# -*- coding: utf-8 -*-
"""label correcting v1.ipynb - Fixed Version

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1luUiE6tRDjaAf_bI0YHSDrhrd_CzflpK
"""

# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from scipy.interpolate import griddata
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d import Axes3D

# ======================== VARIABLES ========================
# 起点和终点的节点 ID（按需修改）
source_id      = 13
destination_id = 17

# ======================== CSV DIRECTORY ========================
# 假设所有 CSV 文件都在当前工作目录
csv_dir = os.getcwd()

# ======================== LOAD SHADOW PROFILES ========================
shadow_profiles = {}
# 查找所有 tree 文件并加载对应 shadow 文件
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # 去掉 '_tree.csv'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    # 读取数值矩阵
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # 如 ['edge','0','1'] → ['0','1']
    shadow_profiles[f"edge_{v}_{u}"] = combined

# ======================== LOAD SOLAR & CLOUD ========================
# 读取实际时间序列
solar_file = os.path.join(csv_dir, 'solar_intensity.csv')
cloud_file = os.path.join(csv_dir, 'cloud_coverage.csv')
if not os.path.exists(solar_file) or not os.path.exists(cloud_file):
    raise FileNotFoundError("缺少 'solar_intensity.csv' 或 'cloud_coverage.csv' 文件")
solar_df = pd.read_csv(solar_file)
cloud_df = pd.read_csv(cloud_file)
R = solar_df.iloc[:,2].astype(float).values
C = cloud_df.iloc[:,2].astype(float).values

# 时间步长
T = R.shape[0]

# ======================== NODE COORDS & ID MAPPING ========================
node_coords = {
    0: (0, 0),         # Downtown center
    1: (0.67, 0.42),   # Northeast of downtown
    2: (0.33, 0.89),   # North of downtown
    3: (-0.48, 0.78),  # Northwest of downtown
    4: (-0.92, 0.31),  # West of downtown
    5: (-0.75, -0.56), # Southwest of downtown
    6: (-0.37, -0.83), # South of downtown
    7: (0.52, -0.69),  # Southeast of downtown
    8: (0.86, -0.27),  # East of downtown
    9: (1.73, 0.86),   # Northeast suburb
    10: (1.26, 1.87),  # North suburb
    11: (-1.32, 1.76), # Northwest suburb
    12: (-1.89, 0.72), # West suburb
    13: (-1.65, -0.94),# Southwest suburb
    14: (-1.18, -1.73),# South suburb
    15: (1.31, -1.62), # Southeast suburb
    16: (1.81, -0.83), # East suburb
    17: (2.23, 0.21),  # Far east
    18: (0.18, 1.76),  # Far north
    19: (0.11, -1.65)  # Far south
}

coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# ======================== RAW EDGES (COORDINATE-BASED) ========================
raw_edges = {
    # ——— from center to first ring ———
    ((0, 0),       (0.67, 0.42)):     40,
    ((0, 0),       (0.33, 0.89)):     55,
    ((0, 0),       (-0.48, 0.78)):    58,
    ((0, 0),       (-0.92, 0.31)):    54,
    ((0, 0),       (-0.75, -0.56)):   52,
    ((0, 0),       (-0.37, -0.83)):   57,
    ((0, 0),       (0.52, -0.69)):    53,
    ((0, 0),       (0.86, -0.27)):    57,

    # ——— first-ring cycle ———
    ((0.67, 0.42), (0.33, 0.89)):     27,
    ((0.33, 0.89), (-0.48, 0.78)):    45,
    ((-0.48, 0.78),(-0.92, 0.31)):    35,
    ((-0.92, 0.31),(-0.75, -0.56)):   54,
    ((-0.75, -0.56),(-0.37, -0.83)):  20,
    ((-0.37, -0.83),(0.52, -0.69)):   54,
    ((0.52, -0.69),(0.86, -0.27)):    26,
    ((0.86, -0.27),(0.67, 0.42)):     42,

    # ——— spokes to second ring ———
    ((0.67, 0.42),  (1.73, 0.86)):    73,   # 1 → 9
    ((0.33, 0.89),  (0.18, 1.76)):    56,   # 2 → 18
    ((-0.48, 0.78), (-1.32, 1.76)):   87,   # 3 → 11
    ((-0.92, 0.31), (-1.89, 0.72)):   59,   # 4 → 12
    ((-0.75, -0.56),(-1.65, -0.94)):  54,   # 5 → 13
    ((-0.37, -0.83),(-1.18, -1.73)):  80,   # 6 → 14
    ((0.52, -0.69), (1.31, -1.62)):   71,   # 7 → 15
    ((0.86, -0.27), (1.81, -0.83)):   70,   # 8 → 16

    # ——— second-ring cycle (adjusted 9–16–17) ———
    ((1.26, 1.87),  (1.73, 0.86)):    74,   # 10 ↔ 9
    ((1.73, 0.86),  (2.23, 0.21)):    46,   # 9 ↔ 17
    ((1.81, -0.83), (2.23, 0.21)):    72,   # 16 ↔ 17
    # (removed original 17 ↔ 9 link)

    # ——— remaining second-ring connections ———
    ((1.26, 1.87),  (0.18, 1.76)):    67,   # 10 ↔ 18
    ((0.18, 1.76),  (-1.32, 1.76)):   89,   # 18 ↔ 11
    ((-1.32, 1.76), (-1.89, 0.72)):   75,   # 11 ↔ 12
    ((-1.89, 0.72), (-1.65, -0.94)):  110,   # 12 ↔ 13
    ((-1.65, -0.94),(-1.18, -1.73)):  55,   # 13 ↔ 14
    ((-1.18, -1.73),(0.11, -1.65)):   74,   # 14 ↔ 19
    ((0.11, -1.65), (1.31, -1.62)):   79,   # 19 ↔ 15
    ((1.31, -1.62), (1.81, -0.83)):   56    # 15 ↔ 16
}



# ======================== DUMMY SHADOW PROFILE FILL ========================
# 对缺失 shadow profile 的边，用 shape=(T,1) 的低值 dummy 填充
for (cu, cv), _ in raw_edges.items():
    u_id = coord_to_id[cu]
    v_id = coord_to_id[cv]
    p = f"edge_{u_id}_{v_id}"; r = f"edge_{v_id}_{u_id}"
    if p not in shadow_profiles:
        dummy = np.full((T,1), -999999.0)
        shadow_profiles[p] = dummy
        shadow_profiles[r] = dummy

# ======================== GRAPH SETUP (ALL EDGES) ========================
graph = {}
for (cu, cv), dist in raw_edges.items():
    u_id = coord_to_id[cu]; v_id = coord_to_id[cv]
    graph.setdefault(u_id, {})[v_id] = dist
    graph.setdefault(v_id, {})[u_id] = dist

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

# ======================== SOLAR EXPOSURE CALC ========================
def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"
    profile = shadow_profiles[key]
    _, K = profile.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        # z < 0 会引入高惩罚：1 - z ≈ 1e6
        z = profile[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path: continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, t0 + steps))
    return results

# ======================== RUN & DISPLAY ========================
print("正在计算从节点 {} 到节点 {} 的多目标最优路径...".format(source_id, destination_id))
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)
print(f"找到 {len(all_sols)} 条路径")

# ======================== CALCULATE EDGE SHADE (补充缺失部分) ========================
# 计算每条边的平均阴影值，用于后续可视化
edge_shade = {}
for (cu, cv), _ in raw_edges.items():
    u_id = coord_to_id[cu]
    v_id = coord_to_id[cv]
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"

    profile = shadow_profiles[key]
    # 如果是dummy数据（全是负值），设为0
    if np.all(profile < 0):
        avg_shade = 0.0
    else:
        # 计算平均阴影值（忽略负值dummy数据）
        valid_values = profile[profile >= 0]
        if len(valid_values) > 0:
            avg_shade = np.mean(valid_values)
        else:
            avg_shade = 0.0

    # 限制在 [0, 1] 范围内
    avg_shade = np.clip(avg_shade, 0, 1)
    edge_shade[(u_id, v_id)] = avg_shade
    edge_shade[(v_id, u_id)] = avg_shade

print(f"已计算 {len(edge_shade)} 条边的平均阴影值")

# ======================== PARETO FRONTIER PLOT (美化) ========================
# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]

print(f"Pareto 最优路径数量: {len(pareto)}")

# ——— Extract the two objective lists ———
ds = [path_obj[1][0] for path_obj in pareto]   # total distances
ss = [path_obj[1][1] for path_obj in pareto]   # total solar exposures

# 1) 计算均值
mean_d = sum(ds) / len(ds)
mean_s = sum(ss) / len(ss)

# 2) 找到三个特殊点索引
idx_shortest   = min(range(len(ds)), key=lambda i: ds[i])
idx_best_shade = min(range(len(ss)), key=lambda i: ss[i])
idx_compromise = min(
    range(len(ds)),
    key=lambda i: (ds[i] - mean_d)**2 + (ss[i] - mean_s)**2
)

# 3) 开始绘图
plt.figure(figsize=(10, 6))

# 所有 Pareto 点
plt.scatter(ds, ss,
            marker='x',
            s=400,
            color='tab:blue',
            label='Pareto-optimal paths')

# 高亮设置：索引, 标签, 颜色, 垂直对齐, 水平对齐
highlights = [
    (idx_shortest,  'Shortest Path',        'tab:green',  'bottom', 'left'),
    (idx_best_shade, 'Best Shade',           'tab:orange', 'top',    'right'),
    (idx_compromise, 'Balanced Compromise',  'tab:red',    'bottom', 'right'),
]

for idx, label, color, va, ha in highlights:
    plt.scatter(ds[idx], ss[idx],
                marker='X',
                s=800,
                color=color,
                linewidths=0.8,   # 控制 "×" 的粗细，数值越小越细
                label=label)

# 标题与坐标轴
plt.title("Pareto Frontier: Distance vs. Solar Exposure", fontsize=16, pad=12)
plt.xlabel("Total Distance", fontsize=14)
plt.ylabel("Total Solar Exposure", fontsize=14)

# 网格
plt.grid(linestyle='--', alpha=0.5)

# 美观的图例：放图内右上角，略偏移避免重叠
leg = plt.legend(
    loc='upper right',        # 图例位置
    fontsize=16,              # 文字大小
    frameon=True,             # 显示边框
    borderpad=1.0,            # 边框内间距
    labelspacing=1.0,         # 条目之间间隔
    handlelength=2.5,         # 图例句柄长度
    handletextpad=1.0,        # 图例句柄与文字间距
    markerscale=0.8           # 放大标记尺寸
)
leg.get_frame().set_edgecolor('gray')   # 边框颜色
leg.get_frame().set_alpha(0.9)          # 背景透明度


# 留出图例空间
plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.savefig('/mnt/user-data/outputs/pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.show()
print("图1: Pareto 前沿图已保存")

# ======================== DRAW & ROUTE ========================
# 假定 graph, node_coords, pareto 都已定义

fig, ax = plt.subplots(figsize=(14, 12))

# 1) 背景路段（浅灰色）
seen = set()
for u, nbrs in graph.items():
    for v, dist in nbrs.items():
        if (v, u) in seen: continue
        seen.add((u, v))
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                color='lightgray', linewidth=1, zorder=1)

# 2) 节点：先画一个白底黑边的大点，再加粗数字
for nid, (x, y) in node_coords.items():
    # 用粉色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为粉色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 3) Pareto 路径：不同线型 + 半透明 + 颜色
cmap = cm.get_cmap('tab10', len(pareto))
line_styles = ['solid', 'dashed', 'dotted', 'dashdot']
for i, (path, (dist, sol)) in enumerate(pareto):
    pts = [node_coords[n] for n in path]
    xs, ys = zip(*pts)
    ax.plot(xs, ys,
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=8,
            linewidth=2.5,
            alpha=0.9,
            color=cmap(i),
            label=f'Path {i+1}: d={dist:.0f}, s={sol:.1f}',
            zorder=4)

# 4) 图例移动到画布外，并优化布局
ax.legend(loc='upper left',
          bbox_to_anchor=(1.02, 1),
          fontsize='small',
          frameon=True)

# 美化
ax.set_title("Pareto-optimal Paths on Pedestrian Network", fontsize=16)
ax.set_xlabel("X Coordinate", fontsize=14)
ax.set_ylabel("Y Coordinate", fontsize=14)
ax.set_aspect('equal', 'box')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.savefig('/mnt/user-data/outputs/path_network.png', dpi=300, bbox_inches='tight')
plt.show()
print("图2: 路径网络图已保存")

# ======================== HEATMAP: EDGE SHADE ========================
plt.figure(figsize=(8, 8))
ax = plt.gca()

for (u, v), shade in edge_shade.items():
    if u < v:
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                linewidth=8,
                solid_capstyle='butt',
                color=plt.cm.YlGn(shade),  # 0→yellow, 1→green
                alpha=0.8)

# 节点
for nid, (x, y) in node_coords.items():
    # 用粉色实心圆 + 黑边
    ax.scatter(x, y,
               s=300,                # 点更大
               facecolor='pink',   # 底色改为粉色
               edgecolor='black',    # 黑边
               linewidth=2,
               zorder=2)
    # 编号文字略微向上偏移，用描边让文字更易读
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    # 添加白色描边（halo）
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 色标
sm = plt.cm.ScalarMappable(cmap='YlGn', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Average Shade (0=no, 1=full)', fontsize=12)

ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Road Segment Shade Heatmap", fontsize=16)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/shade_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("图3: 阴影热力图已保存")

# ======================== CONTOUR MAP ========================
# 1) 计算每个节点的平均 shade 值
node_shade = {nid: [] for nid in node_coords}
for (u, v), shade in edge_shade.items():
    node_shade[u].append(shade)
    node_shade[v].append(shade)
# 只保留平均值
node_vals = {nid: np.mean(vals) if vals else 0.0 for nid, vals in node_shade.items()}

# 2) 准备三角剖分输入
xs = np.array([node_coords[n][0] for n in node_coords])
ys = np.array([node_coords[n][1] for n in node_coords])
zs = np.array([node_vals[n] for n in node_coords])

triang = mtri.Triangulation(xs, ys)

# 3) 绘制等高线填色图
plt.figure(figsize=(10, 8))
contf = plt.tricontourf(triang, zs, levels=12, cmap='YlGn', alpha=0.8)
cont = plt.tricontour(triang, zs, levels=12, colors='k', linewidths=0.5)

# 4) 叠加网络边
for u, nbrs in graph.items():
    for v in nbrs:
        if u < v:
            x1, y1 = node_coords[u]
            x2, y2 = node_coords[v]
            plt.plot([x1, x2], [y1, y2], color='gray', linewidth=1, alpha=0.6)

# 5) 叠加节点
for nid, (x, y) in node_coords.items():
    plt.scatter(x, y, s=40, facecolor='white', edgecolor='black', linewidth=0.8, zorder=3)
    plt.text(x, y+0.02, str(nid), ha='center', va='bottom', fontsize=9)

plt.colorbar(contf, label='Average Shade (0=no, 1=full)')
plt.title("Shadow Intensity Contour Map")
plt.axis('equal')
plt.xticks([]); plt.yticks([])
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/contour_map.png', dpi=300, bbox_inches='tight')
plt.show()
print("图4: 等高线图已保存")

# ======================== 3D SURFACE PLOT (补充缺失部分) ========================
# 生成插值网格
xi = np.linspace(xs.min(), xs.max(), 100)
yi = np.linspace(ys.min(), ys.max(), 100)
Xi, Yi = np.meshgrid(xi, yi)

# 使用 griddata 进行插值
Zi = griddata((xs, ys), zs, (Xi, Yi), method='cubic')

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 光滑曲面，cmap='YlGn'：0→yellow，1→green
surf = ax.plot_surface(
    Xi, Yi, Zi,
    cmap='YlGn',
    edgecolor='none',
    antialiased=True,
    rcount=100, ccount=100  # 更细网格
)

# 原始节点叠加
ax.scatter(xs, ys, zs, color='black', s=30, zorder=5)

# **调整视角**：抬高视点，增大俯仰，让曲面更"倾斜"
ax.view_init(elev=45, azim=-60)

# 坐标轴标签和标题
ax.set_title("Smooth 3D Shadow Surface", fontsize=16)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_zlabel("Average Shade (0–1)")

# 色标
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Average Shade')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/3d_surface.png', dpi=300, bbox_inches='tight')
plt.show()
print("图5: 3D 曲面图已保存")

print("\n所有图表已生成并保存到 /mnt/user-data/outputs/ 目录")
print("修复完成！")

In [ ]:
# -*- coding: utf-8 -*-
"""label correcting v1.ipynb - CSV Input Version

Modified to read network structure from CSV files instead of hardcoded values.
"""

# ======================== IMPORTS ========================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import heapq
import matplotlib.cm as cm
import matplotlib.patheffects as pe
from scipy.interpolate import griddata
import matplotlib.tri as mtri
from mpl_toolkits.mplot3d import Axes3D

# ======================== VARIABLES ========================
# 起点和终点的节点 ID（按需修改）
source_id      = 13
destination_id = 17

# ======================== CSV DIRECTORY ========================
# 假设所有 CSV 文件都在当前工作目录
csv_dir = os.getcwd()

# ======================== LOAD NETWORK STRUCTURE FROM CSV ========================
print("正在加载路网结构...")

# 读取节点信息
nodes_file = os.path.join(csv_dir, 'nodes.csv')
if not os.path.exists(nodes_file):
    raise FileNotFoundError(f"缺少节点文件: {nodes_file}")

nodes_df = pd.read_csv(nodes_file)
print(f"读取到 {len(nodes_df)} 个节点")

# 构建 node_coords 字典: {node_id: (x, y)}
node_coords = {}
for _, row in nodes_df.iterrows():
    node_id = int(row['node_id'])
    x_coord = float(row['x_coord'])
    y_coord = float(row['y_coord'])
    node_coords[node_id] = (x_coord, y_coord)

# 构建反向映射: {(x, y): node_id}
coord_to_id = {coord: nid for nid, coord in node_coords.items()}

# 读取边信息
edges_file = os.path.join(csv_dir, 'edges.csv')
if not os.path.exists(edges_file):
    raise FileNotFoundError(f"缺少边文件: {edges_file}")

edges_df = pd.read_csv(edges_file)
print(f"读取到 {len(edges_df)} 条边")

# 构建 graph 字典: {from_node: {to_node: distance}}
# 注意：这里构建的是无向图，所以每条边都双向添加
graph = {}
for _, row in edges_df.iterrows():
    from_node = int(row['from_node'])
    to_node = int(row['to_node'])
    distance = float(row['distance'])

    # 添加双向边
    graph.setdefault(from_node, {})[to_node] = distance
    graph.setdefault(to_node, {})[from_node] = distance

print(f"路网结构加载完成: {len(node_coords)} 个节点, {len(edges_df)} 条边")

# ======================== LOAD SHADOW PROFILES ========================
print("\n正在加载阴影轮廓...")
shadow_profiles = {}
# 查找所有 tree 文件并加载对应 shadow 文件
shadow_file_count = 0
for fname in sorted(os.listdir(csv_dir)):
    if not fname.endswith('_tree.csv'):
        continue
    prefix      = fname[:-9]  # 去掉 '_tree.csv'
    tree_file   = os.path.join(csv_dir, f"{prefix}_tree.csv")
    shadow_file = os.path.join(csv_dir, f"{prefix}_shadow.csv")
    if not os.path.exists(shadow_file):
        continue
    # 读取数值矩阵
    df_tree   = pd.read_csv(tree_file)
    df_shadow = pd.read_csv(shadow_file)
    tree_vals   = df_tree.select_dtypes(include=[np.number]).values.astype(float)
    shadow_vals = df_shadow.select_dtypes(include=[np.number]).values.astype(float)
    combined = np.maximum(tree_vals, shadow_vals)
    # 存储正反两种前缀，方便无向查找
    shadow_profiles[prefix] = combined
    u, v = prefix.split('_')[1:]  # 如 ['edge','0','1'] → ['0','1']
    shadow_profiles[f"edge_{v}_{u}"] = combined
    shadow_file_count += 1

print(f"加载了 {shadow_file_count} 组阴影轮廓文件")

# ======================== LOAD SOLAR & CLOUD ========================
print("\n正在加载太阳强度和云量数据...")
# 读取实际时间序列
solar_file = os.path.join(csv_dir, 'solar_intensity.csv')
cloud_file = os.path.join(csv_dir, 'cloud_coverage.csv')
if not os.path.exists(solar_file) or not os.path.exists(cloud_file):
    raise FileNotFoundError("缺少 'solar_intensity.csv' 或 'cloud_coverage.csv' 文件")
solar_df = pd.read_csv(solar_file)
cloud_df = pd.read_csv(cloud_file)
R = solar_df.iloc[:,2].astype(float).values
C = cloud_df.iloc[:,2].astype(float).values

# 时间步长
T = R.shape[0]
print(f"时间序列长度: {T}")

# ======================== DUMMY SHADOW PROFILE FILL ========================
# 对缺失 shadow profile 的边，用 shape=(T,1) 的低值 dummy 填充
print("\n正在填充缺失的阴影数据...")
dummy_count = 0
for from_node, neighbors in graph.items():
    for to_node in neighbors.keys():
        p = f"edge_{from_node}_{to_node}"
        r = f"edge_{to_node}_{from_node}"
        if p not in shadow_profiles:
            dummy = np.full((T,1), -999999.0)
            shadow_profiles[p] = dummy
            shadow_profiles[r] = dummy
            dummy_count += 1

if dummy_count > 0:
    print(f"填充了 {dummy_count // 2} 条边的缺失阴影数据（使用dummy值）")
else:
    print("所有边都有阴影数据")

# ======================== HELPER FUNCS ========================
def is_dominated(a, b):
    return all(x <= y for x, y in zip(a, b)) and any(x < y for x, y in zip(a, b))

def add_label_check(label_dict, node, obj):
    return not any(is_dominated(obj, old) for old in label_dict[node])

# ======================== SOLAR EXPOSURE CALC ========================
def calculate_solar_exposure(u_id, v_id, start_time):
    key = f"edge_{u_id}_{v_id}"
    if key not in shadow_profiles:
        key = f"edge_{v_id}_{u_id}"
    profile = shadow_profiles[key]
    _, K = profile.shape
    total = 0.0
    for k in range(K):
        t = start_time + k
        if t >= T: break
        # z < 0 会引入高惩罚：1 - z ≈ 1e6
        z = profile[t, k]
        if z >= 1:
          z = 1
        total += R[t] * (1 - C[t]) * (1 - z)
    return total, K

# ======================== MULTI-OBJECTIVE SEARCH ========================
def bi_objective_label_correcting(graph, source, dest):
    queue      = [(0.0, 0, [source], [0.0, 0.0], 0)]
    label_dict = {n: [] for n in graph}
    results    = []
    uid        = 0

    while queue:
        _, _, path, obj, t0 = heapq.heappop(queue)
        u = path[-1]
        if not add_label_check(label_dict, u, obj):
            continue
        label_dict[u].append(obj)
        if u == dest:
            results.append((path, obj))
            continue
        for v, dist in graph[u].items():
            if v in path: continue
            solar, steps = calculate_solar_exposure(u, v, t0)
            new_obj = [obj[0] + dist, obj[1] + solar]
            uid += 1
            heapq.heappush(queue, (sum(new_obj), uid, path + [v], new_obj, t0 + steps))
    return results

# ======================== RUN & DISPLAY ========================
print("\n" + "="*60)
print(f"正在计算从节点 {source_id} 到节点 {destination_id} 的多目标最优路径...")
print("="*60)
all_sols = bi_objective_label_correcting(graph, source_id, destination_id)
print(f"找到 {len(all_sols)} 条路径")

# ======================== CALCULATE EDGE SHADE ========================
print("\n正在计算边的平均阴影值...")
edge_shade = {}
for from_node, neighbors in graph.items():
    for to_node in neighbors.keys():
        if (from_node, to_node) in edge_shade:
            continue  # 已经处理过这条边

        key = f"edge_{from_node}_{to_node}"
        if key not in shadow_profiles:
            key = f"edge_{to_node}_{from_node}"

        profile = shadow_profiles[key]
        # 如果是dummy数据（全是负值），设为0
        if np.all(profile < 0):
            avg_shade = 0.0
        else:
            # 计算平均阴影值（忽略负值dummy数据）
            valid_values = profile[profile >= 0]
            if len(valid_values) > 0:
                avg_shade = np.mean(valid_values)
            else:
                avg_shade = 0.0

        # 限制在 [0, 1] 范围内
        avg_shade = np.clip(avg_shade, 0, 1)
        edge_shade[(from_node, to_node)] = avg_shade
        edge_shade[(to_node, from_node)] = avg_shade

print(f"已计算 {len(edge_shade) // 2} 条边的平均阴影值")

# ======================== PARETO FRONTIER PLOT ========================
print("\n正在生成Pareto前沿图...")
# 过滤 Pareto
pareto = [s for s in all_sols if not any(
    other[1][0] <= s[1][0] and other[1][1] <= s[1][1] and
    (other[1][0] < s[1][0] or other[1][1] < s[1][1])
    for other in all_sols
)]

print(f"Pareto 最优路径数量: {len(pareto)}")

# 打印每条Pareto最优路径的详细信息
print("\nPareto最优路径详情:")
for i, (path, (dist, solar)) in enumerate(pareto):
    print(f"  路径 {i+1}: {' -> '.join(map(str, path))}")
    print(f"    总距离: {dist:.2f}, 总太阳暴露: {solar:.2f}")

# ——— Extract the two objective lists ———
ds = [path_obj[1][0] for path_obj in pareto]   # total distances
ss = [path_obj[1][1] for path_obj in pareto]   # total solar exposures

# 1) 计算均值
mean_d = sum(ds) / len(ds)
mean_s = sum(ss) / len(ss)

# 2) 找到三个特殊点索引
idx_shortest   = min(range(len(ds)), key=lambda i: ds[i])
idx_best_shade = min(range(len(ss)), key=lambda i: ss[i])
idx_compromise = min(
    range(len(ds)),
    key=lambda i: (ds[i] - mean_d)**2 + (ss[i] - mean_s)**2
)

# 3) 开始绘图
plt.figure(figsize=(10, 6))

# 所有 Pareto 点
plt.scatter(ds, ss,
            marker='x',
            s=400,
            color='tab:blue',
            label='Pareto-optimal paths')

# 高亮设置：索引, 标签, 颜色, 垂直对齐, 水平对齐
highlights = [
    (idx_shortest,  'Shortest Path',        'tab:green',  'bottom', 'left'),
    (idx_best_shade, 'Best Shade',           'tab:orange', 'top',    'right'),
    (idx_compromise, 'Balanced Compromise',  'tab:red',    'bottom', 'right'),
]

for idx, label, color, va, ha in highlights:
    plt.scatter(ds[idx], ss[idx],
                marker='X',
                s=800,
                color=color,
                linewidths=0.8,
                label=label)

# 标题与坐标轴
plt.title("Pareto Frontier: Distance vs. Solar Exposure", fontsize=16, pad=12)
plt.xlabel("Total Distance", fontsize=14)
plt.ylabel("Total Solar Exposure", fontsize=14)

# 网格
plt.grid(linestyle='--', alpha=0.5)

# 美观的图例
leg = plt.legend(
    loc='upper right',
    fontsize=16,
    frameon=True,
    borderpad=1.0,
    labelspacing=1.0,
    handlelength=2.5,
    handletextpad=1.0,
    markerscale=0.8
)
leg.get_frame().set_edgecolor('gray')
leg.get_frame().set_alpha(0.9)

plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.savefig('/mnt/user-data/outputs/pareto_frontier.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图1: Pareto 前沿图已保存")

# ======================== DRAW & ROUTE ========================
print("\n正在生成路径网络图...")

fig, ax = plt.subplots(figsize=(14, 12))

# 1) 背景路段（浅灰色）
seen = set()
for u, nbrs in graph.items():
    for v, dist in nbrs.items():
        if (v, u) in seen: continue
        seen.add((u, v))
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                color='lightgray', linewidth=1, zorder=1)

# 2) 节点
for nid, (x, y) in node_coords.items():
    ax.scatter(x, y,
               s=300,
               facecolor='pink',
               edgecolor='black',
               linewidth=2,
               zorder=2)
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 3) Pareto 路径
cmap = cm.get_cmap('tab10', len(pareto))
line_styles = ['solid', 'dashed', 'dotted', 'dashdot']
for i, (path, (dist, sol)) in enumerate(pareto):
    pts = [node_coords[n] for n in path]
    xs, ys = zip(*pts)
    ax.plot(xs, ys,
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=8,
            linewidth=2.5,
            alpha=0.9,
            color=cmap(i),
            label=f'Path {i+1}: d={dist:.0f}, s={sol:.1f}',
            zorder=4)

# 4) 图例
ax.legend(loc='upper left',
          bbox_to_anchor=(1.02, 1),
          fontsize='small',
          frameon=True)

ax.set_title("Pareto-optimal Paths on Pedestrian Network", fontsize=16)
ax.set_xlabel("X Coordinate", fontsize=14)
ax.set_ylabel("Y Coordinate", fontsize=14)
ax.set_aspect('equal', 'box')
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout(rect=(0, 0, 0.85, 1))
plt.savefig('/mnt/user-data/outputs/path_network.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图2: 路径网络图已保存")

# ======================== HEATMAP: EDGE SHADE ========================
print("\n正在生成阴影热力图...")

plt.figure(figsize=(8, 8))
ax = plt.gca()

for (u, v), shade in edge_shade.items():
    if u < v:
        x1, y1 = node_coords[u]
        x2, y2 = node_coords[v]
        ax.plot([x1, x2], [y1, y2],
                linewidth=8,
                solid_capstyle='butt',
                color=plt.cm.YlGn(shade),
                alpha=0.8)

# 节点
for nid, (x, y) in node_coords.items():
    ax.scatter(x, y,
               s=300,
               facecolor='pink',
               edgecolor='black',
               linewidth=2,
               zorder=2)
    txt = ax.text(x, y + 0.1, str(nid),
                  ha='center', va='bottom',
                  fontsize=14,
                  fontweight='bold',
                  color='black',
                  zorder=3)
    txt.set_path_effects([
        pe.Stroke(linewidth=3, foreground='white'),
        pe.Normal()
    ])

# 色标
sm = plt.cm.ScalarMappable(cmap='YlGn', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Average Shade (0=no, 1=full)', fontsize=12)

ax.set_aspect('equal')
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Road Segment Shade Heatmap", fontsize=16)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/shade_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图3: 阴影热力图已保存")

# ======================== CONTOUR MAP ========================
print("\n正在生成等高线图...")

# 1) 计算每个节点的平均 shade 值
node_shade = {nid: [] for nid in node_coords}
for (u, v), shade in edge_shade.items():
    node_shade[u].append(shade)
    node_shade[v].append(shade)
node_vals = {nid: np.mean(vals) if vals else 0.0 for nid, vals in node_shade.items()}

# 2) 准备三角剖分输入
xs = np.array([node_coords[n][0] for n in node_coords])
ys = np.array([node_coords[n][1] for n in node_coords])
zs = np.array([node_vals[n] for n in node_coords])

triang = mtri.Triangulation(xs, ys)

# 3) 绘制等高线填色图
plt.figure(figsize=(10, 8))
contf = plt.tricontourf(triang, zs, levels=12, cmap='YlGn', alpha=0.8)
cont = plt.tricontour(triang, zs, levels=12, colors='k', linewidths=0.5)

# 4) 叠加网络边
for u, nbrs in graph.items():
    for v in nbrs:
        if u < v:
            x1, y1 = node_coords[u]
            x2, y2 = node_coords[v]
            plt.plot([x1, x2], [y1, y2], color='gray', linewidth=1, alpha=0.6)

# 5) 叠加节点
for nid, (x, y) in node_coords.items():
    plt.scatter(x, y, s=40, facecolor='white', edgecolor='black', linewidth=0.8, zorder=3)
    plt.text(x, y+0.02, str(nid), ha='center', va='bottom', fontsize=9)

plt.colorbar(contf, label='Average Shade (0=no, 1=full)')
plt.title("Shadow Intensity Contour Map")
plt.axis('equal')
plt.xticks([]); plt.yticks([])
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/contour_map.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图4: 等高线图已保存")

# ======================== 3D SURFACE PLOT ========================
print("\n正在生成3D曲面图...")

# 生成插值网格
xi = np.linspace(xs.min(), xs.max(), 100)
yi = np.linspace(ys.min(), ys.max(), 100)
Xi, Yi = np.meshgrid(xi, yi)

# 使用 griddata 进行插值
Zi = griddata((xs, ys), zs, (Xi, Yi), method='cubic')

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 光滑曲面
surf = ax.plot_surface(
    Xi, Yi, Zi,
    cmap='YlGn',
    edgecolor='none',
    antialiased=True,
    rcount=100, ccount=100
)

# 原始节点叠加
ax.scatter(xs, ys, zs, color='black', s=30, zorder=5)

# 调整视角
ax.view_init(elev=45, azim=-60)

# 标题和标签
ax.set_title("Smooth 3D Shadow Surface", fontsize=16)
ax.set_xlabel("X Coordinate")
ax.set_ylabel("Y Coordinate")
ax.set_zlabel("Average Shade (0–1)")

# 色标
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, label='Average Shade')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/3d_surface.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ 图5: 3D 曲面图已保存")

print("\n" + "="*60)
print("所有图表已生成并保存到 /mnt/user-data/outputs/ 目录")
print("程序运行完成！")
print("="*60)

In [ ]:
"""
快速测试脚本 - 下载Times Square周边小区域
用于快速验证系统是否正常工作

运行时间: ~10-20秒
节点数: ~100-200个
"""

import osmnx as ox
import pandas as pd
import matplotlib.pyplot as plt

print("="*60)
print("快速测试: 下载Times Square周边路网")
print("="*60)

# Times Square周边小区域（约0.5km x 0.5km）
BBOX = {
    'north': 40.7600,
    'south': 40.7550,
    'east': -73.9830,
    'west': -73.9880
}

OUTPUT_DIR = '/mnt/user-data/outputs/'

try:
    print("\n[1/5] 正在下载路网数据...")
    G = ox.graph_from_bbox(
        north=BBOX['north'],
        south=BBOX['south'],
        east=BBOX['east'],
        west=BBOX['west'],
        network_type='walk',
        simplify=True
    )
    print(f"✓ 下载完成: {G.number_of_nodes()} 节点, {G.number_of_edges()} 条边")

    print("\n[2/5] 转换坐标系统...")
    G_proj = ox.project_graph(G)

    # 提取节点
    nodes_data = []
    for node, data in G_proj.nodes(data=True):
        nodes_data.append({
            'node_id': node,
            'x_coord': data['x'],
            'y_coord': data['y']
        })
    nodes_df = pd.DataFrame(nodes_data)

    # 归一化坐标
    x_min, x_max = nodes_df['x_coord'].min(), nodes_df['x_coord'].max()
    y_min, y_max = nodes_df['y_coord'].min(), nodes_df['y_coord'].max()
    scale = 10 / max(x_max - x_min, y_max - y_min)
    nodes_df['x_coord'] = (nodes_df['x_coord'] - x_min) * scale - 5
    nodes_df['y_coord'] = (nodes_df['y_coord'] - y_min) * scale - 5
    nodes_df['description'] = nodes_df['node_id'].apply(lambda x: f"Node {x}")
    print(f"✓ 坐标处理完成")

    print("\n[3/5] 提取边信息...")
    edges_data = []
    for u, v, key, data in G_proj.edges(keys=True, data=True):
        edges_data.append({
            'from_node': u,
            'to_node': v,
            'distance': data.get('length', 0),
            'description': data.get('name', 'Unnamed')
        })
    edges_df = pd.DataFrame(edges_data)
    print(f"✓ 提取 {len(edges_df)} 条边")

    print("\n[4/5] 保存CSV文件...")
    nodes_file = OUTPUT_DIR + 'nodes_test.csv'
    edges_file = OUTPUT_DIR + 'edges_test.csv'
    nodes_df[['node_id', 'x_coord', 'y_coord', 'description']].to_csv(nodes_file, index=False)
    edges_df.to_csv(edges_file, index=False)
    print(f"✓ 保存到: {nodes_file}")
    print(f"         {edges_file}")

    print("\n[5/5] 生成可视化...")
    fig, ax = plt.subplots(figsize=(10, 10))

    # 绘制边
    for _, edge in edges_df.iterrows():
        n1 = nodes_df[nodes_df['node_id'] == edge['from_node']].iloc[0]
        n2 = nodes_df[nodes_df['node_id'] == edge['to_node']].iloc[0]
        ax.plot([n1['x_coord'], n2['x_coord']],
                [n1['y_coord'], n2['y_coord']],
                'gray', linewidth=0.8, alpha=0.6)

    # 绘制节点
    ax.scatter(nodes_df['x_coord'], nodes_df['y_coord'],
               s=20, color='red', alpha=0.7, zorder=2)

    ax.set_title("Times Square Area - Test Network", fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

    viz_file = OUTPUT_DIR + 'network_test_viz.png'
    plt.savefig(viz_file, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ 可视化保存到: {viz_file}")

    print("\n" + "="*60)
    print("✅ 测试成功!")
    print("="*60)
    print(f"节点数: {len(nodes_df)}")
    print(f"边数: {len(edges_df)}")
    print(f"平均距离: {edges_df['distance'].mean():.1f} 米")
    print(f"\n示例节点ID (可用作起点/终点):")
    for i, nid in enumerate(nodes_df['node_id'].head(5)):
        print(f"  {i+1}. {nid}")

    print(f"\n现在可以在路径规划脚本中使用:")
    print(f"  source_id = {nodes_df['node_id'].iloc[0]}")
    print(f"  destination_id = {nodes_df['node_id'].iloc[-1]}")

except Exception as e:
    print(f"\n❌ 测试失败: {e}")
    print("\n可能的原因:")
    print("1. 网络连接问题")
    print("2. OSMnx未正确安装")
    print("3. OpenStreetMap服务暂时不可用")
    print("\n解决方案:")
    print("pip install osmnx --break-system-packages --upgrade")

In [ ]:
pip install osmnx --break-system-packages --upgrade

In [ ]:
pip install osmnx networkx pandas matplotlib --break-system-packages

In [ ]:
# ========== Colab OSMnx 测试 - 一键运行 ==========

import sys

print("="*70)
print("🌐 Google Colab OSMnx 测试")
print("="*70)

# 1. 安装依赖
print("\n[1/6] 检查依赖...")
try:
    import osmnx as ox
    print(f"✅ OSMnx {ox.__version__}")
except:
    print("⏳ 安装 osmnx...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "osmnx", "-q"])
    import osmnx as ox
    print(f"✅ OSMnx {ox.__version__} 安装完成")

import pandas as pd
import matplotlib.pyplot as plt
import math

# 2. 下载数据（自动尝试多种方法）
print("\n[2/6] 下载Times Square周边路网...")

G = None

# 方法1: graph_from_point (最稳定)
try:
    center = (40.7575, -73.9855)  # Times Square
    radius = 500  # 500米
    print(f"  尝试方法: graph_from_point (半径{radius}米)")
    G = ox.graph_from_point(center, dist=radius, network_type='walk')
    print(f"✅ 成功! 节点: {G.number_of_nodes()}, 边: {G.number_of_edges()}")
    method = "graph_from_point"
except Exception as e:
    print(f"❌ 失败: {str(e)[:50]}")

# 方法2: graph_from_bbox (新版API)
if G is None:
    try:
        print(f"  尝试方法: graph_from_bbox (位置参数)")
        G = ox.graph_from_bbox(40.76, 40.755, -73.983, -73.988, network_type='walk')
        print(f"✅ 成功! 节点: {G.number_of_nodes()}, 边: {G.number_of_edges()}")
        method = "graph_from_bbox (位置参数)"
    except Exception as e:
        print(f"❌ 失败: {str(e)[:50]}")

# 方法3: graph_from_bbox (旧版API)
if G is None:
    try:
        print(f"  尝试方法: graph_from_bbox (关键字参数)")
        G = ox.graph_from_bbox(north=40.76, south=40.755, east=-73.983, west=-73.988, network_type='walk')
        print(f"✅ 成功! 节点: {G.number_of_nodes()}, 边: {G.number_of_edges()}")
        method = "graph_from_bbox (关键字参数)"
    except Exception as e:
        print(f"❌ 失败: {str(e)[:50]}")

if G is None:
    raise Exception("所有方法都失败了，请检查网络连接")

# 3. 转换坐标
print(f"\n[3/6] 处理坐标...")
try:
    G_proj = ox.project_graph(G)
except:
    G_proj = G

# 4. 提取节点
print(f"[4/6] 提取节点...")
nodes = []
for node, data in G_proj.nodes(data=True):
    x = data.get('x', data.get('lon', 0))
    y = data.get('y', data.get('lat', 0))
    nodes.append({'node_id': node, 'x_coord': float(x), 'y_coord': float(y)})

nodes_df = pd.DataFrame(nodes)

# 归一化坐标
x_mean, y_mean = nodes_df['x_coord'].mean(), nodes_df['y_coord'].mean()
x_std, y_std = nodes_df['x_coord'].std(), nodes_df['y_coord'].std()
nodes_df['x_coord'] = (nodes_df['x_coord'] - x_mean) / x_std * 5
nodes_df['y_coord'] = (nodes_df['y_coord'] - y_mean) / y_std * 5
nodes_df['description'] = 'Node'

print(f"✅ {len(nodes_df)} 个节点")

# 5. 提取边
print(f"[5/6] 提取边...")
edges = []
try:
    for u, v, data in G_proj.edges(data=True):
        edges.append({
            'from_node': u,
            'to_node': v,
            'distance': data.get('length', 0),
            'description': 'Road'
        })
except:
    for u, v, k, data in G_proj.edges(keys=True, data=True):
        edges.append({
            'from_node': u,
            'to_node': v,
            'distance': data.get('length', 0),
            'description': 'Road'
        })

edges_df = pd.DataFrame(edges)
print(f"✅ {len(edges_df)} 条边")

# 6. 保存CSV
print(f"[6/6] 保存文件...")
nodes_df[['node_id', 'x_coord', 'y_coord', 'description']].to_csv('nodes.csv', index=False)
edges_df.to_csv('edges.csv', index=False)
print(f"✅ nodes.csv 和 edges.csv 已保存")

# 7. 可视化
fig, ax = plt.subplots(figsize=(12, 10))
for _, e in edges_df.head(300).iterrows():
    try:
        n1 = nodes_df[nodes_df['node_id'] == e['from_node']].iloc[0]
        n2 = nodes_df[nodes_df['node_id'] == e['to_node']].iloc[0]
        ax.plot([n1['x_coord'], n2['x_coord']], [n1['y_coord'], n2['y_coord']],
                'gray', linewidth=0.5, alpha=0.5)
    except: pass

ax.scatter(nodes_df['x_coord'], nodes_df['y_coord'], s=10, color='red', alpha=0.6)
ax.set_title(f"Times Square Road Network\n方法: {method}", fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('network.png', dpi=150, bbox_inches='tight')
plt.show()

# 8. 总结
print("\n" + "="*70)
print("🎉 完成!")
print("="*70)
print(f"✅ 方法: {method}")
print(f"✅ 节点: {len(nodes_df)}, 边: {len(edges_df)}")
print(f"\n示例节点ID:")
for i, nid in enumerate(nodes_df['node_id'].head(3)):
    print(f"  {nid}")
print(f"\n💡 用于路径规划:")
print(f"  source_id = {nodes_df['node_id'].iloc[0]}")
print(f"  destination_id = {nodes_df['node_id'].iloc[-1]}")
print("\n📁 文件: nodes.csv, edges.csv, network.png")

In [ ]:
# 下载生成的文件
from google.colab import files
files.download('nodes.csv')
files.download('edges.csv')
files.download('network.png')

In [ ]:
# ========== 完整Manhattan路网下载 ==========
# 预计时间: 5-10分钟 | 节点: 40,000-60,000

import sys, time
start_time = time.time()

print("🗽 下载Manhattan完整路网...")

# 1. 安装依赖
try:
    import osmnx as ox
    print(f"✅ OSMnx {ox.__version__}")
except:
    print("⏳ 安装osmnx...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "osmnx", "-q"])
    import osmnx as ox

import pandas as pd
import matplotlib.pyplot as plt

# 2. 下载Manhattan数据
print("\n⏳ 下载中 (需要5-10分钟)...")
try:
    G = ox.graph_from_place(
        "Manhattan, New York City, New York, USA",
        network_type='walk',
        simplify=True
    )
    print(f"✅ 下载完成! 节点: {G.number_of_nodes():,}, 边: {G.number_of_edges():,}")
except Exception as e:
    print(f"❌ 使用地点名称失败，尝试边界框...")
    G = ox.graph_from_bbox(
        40.8820, 40.7000, -73.9070, -74.0200,  # Manhattan边界
        network_type='walk',
        simplify=True
    )
    print(f"✅ 下载完成! 节点: {G.number_of_nodes():,}, 边: {G.number_of_edges():,}")

# 3. 投影坐标
print("⏳ 处理坐标...")
try:
    G_proj = ox.project_graph(G)
except:
    G_proj = G

# 4. 提取节点
print("⏳ 提取节点...")
nodes = []
count = 0
for node, data in G_proj.nodes(data=True):
    x = data.get('x', data.get('lon', 0))
    y = data.get('y', data.get('lat', 0))
    nodes.append({
        'node_id': node,
        'x_coord': float(x),
        'y_coord': float(y),
        'longitude': data.get('lon', x),
        'latitude': data.get('lat', y)
    })
    count += 1
    if count % 5000 == 0:
        print(f"  已处理 {count:,} 个节点...")

nodes_df = pd.DataFrame(nodes)

# 归一化坐标
x_min, x_max = nodes_df['x_coord'].min(), nodes_df['x_coord'].max()
y_min, y_max = nodes_df['y_coord'].min(), nodes_df['y_coord'].max()
nodes_df['x_coord'] = (nodes_df['x_coord'] - x_min) / (x_max - x_min) * 20 - 10
nodes_df['y_coord'] = (nodes_df['y_coord'] - y_min) / (y_max - y_min) * 20 - 10
nodes_df['description'] = 'Node'

print(f"✅ {len(nodes_df):,} 个节点")

# 5. 提取边
print("⏳ 提取边...")
edges = []
count = 0
try:
    for u, v, data in G_proj.edges(data=True):
        edges.append({
            'from_node': u,
            'to_node': v,
            'distance': data.get('length', 0),
            'description': data.get('name', 'Road')
        })
        count += 1
        if count % 10000 == 0:
            print(f"  已处理 {count:,} 条边...")
except:
    for u, v, k, data in G_proj.edges(keys=True, data=True):
        edges.append({
            'from_node': u,
            'to_node': v,
            'distance': data.get('length', 0),
            'description': data.get('name', 'Road')
        })
        count += 1
        if count % 10000 == 0:
            print(f"  已处理 {count:,} 条边...")

edges_df = pd.DataFrame(edges)
print(f"✅ {len(edges_df):,} 条边")

# 6. 保存文件
print("\n⏳ 保存CSV文件...")
nodes_df[['node_id', 'x_coord', 'y_coord', 'description']].to_csv('manhattan_nodes.csv', index=False)
edges_df.to_csv('manhattan_edges.csv', index=False)
nodes_df.to_csv('manhattan_nodes_full.csv', index=False)

print(f"✅ manhattan_nodes.csv")
print(f"✅ manhattan_edges.csv")
print(f"✅ manhattan_nodes_full.csv (含经纬度)")

# 7. 简单可视化
print("\n⏳ 生成可视化...")
fig, ax = plt.subplots(figsize=(12, 16))
sample_edges = edges_df.iloc[::10]
for _, e in sample_edges.iterrows():
    try:
        n1 = nodes_df[nodes_df['node_id'] == e['from_node']].iloc[0]
        n2 = nodes_df[nodes_df['node_id'] == e['to_node']].iloc[0]
        ax.plot([n1['x_coord'], n2['x_coord']], [n1['y_coord'], n2['y_coord']],
                'gray', linewidth=0.3, alpha=0.4)
    except: pass

sample_nodes = nodes_df.iloc[::20]
ax.scatter(sample_nodes['x_coord'], sample_nodes['y_coord'], s=2, color='red', alpha=0.5)
ax.set_title(f"Manhattan Road Network\n{len(nodes_df):,} nodes, {len(edges_df):,} edges",
             fontsize=16, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('manhattan_network.png', dpi=150, bbox_inches='tight')
plt.show()

# 8. 总结
total_time = time.time() - start_time
print("\n" + "="*70)
print("🎉 完成!")
print("="*70)
print(f"✅ 节点: {len(nodes_df):,}, 边: {len(edges_df):,}")
print(f"✅ 用时: {total_time/60:.1f} 分钟")
print(f"✅ 平均距离: {edges_df['distance'].mean():.1f} 米")
print(f"\n示例节点ID:")
for i, nid in enumerate(nodes_df['node_id'].head(5)):
    print(f"  {nid}")
print(f"\n💡 用于路径规划:")
print(f"  source_id = {nodes_df['node_id'].iloc[0]}")
print(f"  destination_id = {nodes_df['node_id'].iloc[-1]}")

In [ ]:
from google.colab import files
files.download('manhattan_network.png')

In [ ]:
from google.colab import files
files.download('manhattan_nodes.csv')
files.download('manhattan_edges.csv')